# Dhruvesh Deep Feature Analysis

This notebook follows Weikun's feature diagnostics and creates presentation-ready tables and visuals. It explains each step in plain language.

Important scope: the analysis reuses Weikun's saved 2020 outputs. It does not modify his notebooks or claim an untouched final test.

## Cell 1 — Run the reproducible analysis

This runs the companion script. The script is divided into numbered cell-style sections and saves all tables and editable SVG visuals under `Dhruvesh/feature_analysis_outputs/`.

In [1]:
from pathlib import Path
import runpy

repo = Path.cwd()
if repo.name != 'practicum-project':
    repo = Path('/Users/dhruvesh/Desktop/Fall 2026/Analytics/practicum-project')
runpy.run_path(str(repo / 'Dhruvesh' / 'feature_analysis_deep.py'))

Generated feature-analysis outputs in: /Users/dhruvesh/Desktop/Fall 2026/Analytics/practicum-project/Dhruvesh/feature_analysis_outputs
Generated 6 presentation-ready SVG figures


{'__name__': '<run_path>',
 '__doc__': "Dhruvesh feature-analysis companion to Weikun's diagnostics.\n\nThis script intentionally reuses Weikun's saved, verified outputs instead of\nrefitting the models. Each numbered section is written like a notebook cell:\nthe comment explains what the section does in plain language, and the outputs\nare saved for later presentation work.\n",
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/Users/dhruvesh/Desktop/Fall 2026/Analytics/practicum-project/Dhruvesh/feature_analysis_deep.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built

## Cell 2 — What features entered the models?

We use 34 candidate features in three main families: local case history, commuting pressure, and calendar controls. The important point is that several columns describe similar information, especially the 7-day and 14-day case averages.

In [2]:
import pandas as pd
from pathlib import Path
out = repo / 'Dhruvesh' / 'feature_analysis_outputs'
pd.read_csv(out / 'feature_inventory.csv').drop_duplicates('feature_group')

,feature_group,feature,feature_count
0,calendar,target_month_2,17
1,commuting_pressure,nj_covid_pressure_7d,2
2,local_case_history,cases_today,15


## Cell 3 — Which individual features did tree models use most?

Training importance answers: “Which inputs did the model use most often while fitting?” It does not prove that a feature caused cases. We look at XGBoost and Random Forest separately because their importance definitions differ.

In [3]:
top = pd.read_csv(out / 'top10_features_by_model_plain_english.csv')
top[top['model'].isin(['Random Forest', 'XGBoost'])].sort_values(['model', 'mean'], ascending=[True, False]).head(10)

,model,feature,feature_group,mean,median,std,nonzero_share,rank,plain_feature
0,Random Forest,cases_mean_7d,local_case_history,0.592223,0.572638,0.241817,1.0,1.0,7-day average cases
1,Random Forest,cases_mean_14d,local_case_history,0.213620,0.189314,0.160907,1.0,2.0,14-day average cases
2,Random Forest,cases_today,local_case_history,0.039269,0.008694,0.077084,1.0,3.0,Current cases
3,Random Forest,case_rate_14d_per_100k,local_case_history,0.023530,0.011932,0.021462,1.0,4.0,14-day case rate
4,Random Forest,nj_covid_pressure_7d,commuting_pressure,0.021513,0.011764,0.018941,1.0,5.0,7-day commuting pressure
5,Random Forest,cases_lag_1,local_case_history,0.016372,0.006161,0.033306,1.0,6.0,Cases 1 day earlier
6,Random Forest,case_std_14d,local_case_history,0.015429,0.012388,0.012506,1.0,7.0,14-day variability
7,Random Forest,nj_covid_pressure_14d,commuting_pressure,0.015183,0.008719,0.012424,1.0,8.0,14-day commuting pressure
8,Random Forest,case_rate_7d_per_100k,local_case_history,0.014165,0.008181,0.012192,1.0,9.0,7-day case rate
9,Random Forest,case_rate_today_per_100k,local_case_history,0.008221,0.004439,0.006331,1.0,10.0,case_rate_today_per_100k


### How to explain the output

Both models place the 7-day average cases, 14-day average cases, and current cases near the top. That means recent local activity is the strongest signal. The model is mostly learning persistence: places with recent cases tend to have cases again soon afterward.

## Cell 4 — Does removing a feature group hurt prediction?

Ablation is a stronger predictive test than importance. We remove one feature group, refit the model, and compare MAE with the full model. Positive values mean the model became worse after removal.

In [4]:
ablation = pd.read_csv(out / 'ablation_summary_clean.csv')
ablation[['model', 'drop_group', 'FULL_MAE', 'DROP_MAE', 'relative_increase_pct', 'ci95_low', 'ci95_high']]

,model,drop_group,FULL_MAE,DROP_MAE,relative_increase_pct,ci95_low,ci95_high
0,Dynamic Panel FE,local_case_history,0.774283,0.956653,23.553383,0.092043,0.288168
1,Dynamic Panel FE,commuting_pressure,0.774283,0.767056,-0.933342,-0.027461,0.010094
2,Dynamic Panel FE,calendar,0.774283,0.778660,0.565338,-0.004488,0.015547
3,XGBoost,local_case_history,0.754299,0.780394,3.459554,0.015151,0.036881
4,XGBoost,commuting_pressure,0.754299,0.756246,0.258157,-0.000759,0.004819
5,XGBoost,calendar,0.754299,0.756705,0.318900,-0.002897,0.008597
6,Random Forest,local_case_history,0.751885,1.415063,88.202006,0.277334,1.120811
7,Random Forest,commuting_pressure,0.751885,0.749545,-0.311237,-0.006722,0.001597
8,Random Forest,calendar,0.751885,0.748764,-0.415206,-0.007348,0.000154


### How to explain the output

Removing local case history increases error substantially: about 23.6% for Dynamic Panel, 3.5% for XGBoost, and 88.2% for Random Forest. Removing commuting pressure or calendar features changes MAE by less than about 1%, and the uncertainty intervals cross zero.

This does not mean commuting is unimportant in the real world. It means that, after recent local case history is already included, commuting pressure adds little measurable improvement in this particular validation.

## Cell 5 — Is the feature story stable across forecast horizons?

We compare 1-day, 2-day, and 3-day forecasts. A feature that stays near the top at all horizons is a more stable result than one that appears at only one horizon.

In [5]:
stability = pd.read_csv(out / 'horizon_stability.csv')
stability.sort_values(['model', 'top5_horizons', 'mean_horizon_importance'], ascending=[True, False, False]).head(20)

,model,feature,top5_horizons,mean_horizon_importance,feature_group
0,Random Forest,cases_mean_7d,3,0.592223,local_case_history
1,Random Forest,cases_mean_14d,3,0.213620,local_case_history
2,Random Forest,cases_today,3,0.039269,local_case_history
3,Random Forest,case_rate_14d_per_100k,3,0.023530,local_case_history
4,Random Forest,nj_covid_pressure_7d,3,0.021513,commuting_pressure
5,Random Forest,cases_lag_1,0,0.016372,local_case_history
6,Random Forest,case_std_14d,0,0.015429,local_case_history
7,Random Forest,nj_covid_pressure_14d,0,0.015183,commuting_pressure
8,Random Forest,case_rate_7d_per_100k,0,0.014165,local_case_history
9,Random Forest,case_rate_today_per_100k,0,0.008221,local_case_history


## Cell 6 — What did feature selection find?

Forward, backward, and stepwise selection search for a smaller feature set. Agreement across methods is useful; disagreement usually means correlated features can substitute for each other.

In [6]:
selection = pd.read_csv(out / 'feature_selection_consensus.csv')
selection[['feature', 'feature_group', 'selected_count', 'selection_consensus']].sort_values(['selected_count', 'feature'], ascending=[False, True])

,feature,feature_group,selected_count,selection_consensus
7,cases_mean_7d,local_case_history,3,all three methods
8,cases_today,local_case_history,3,all three methods
6,nj_covid_pressure_7d,commuting_pressure,3,all three methods
0,target_weekday_2,calendar,3,all three methods
1,target_weekday_3,calendar,3,all three methods
2,target_weekday_4,calendar,3,all three methods
3,target_weekday_5,calendar,3,all three methods
4,target_weekday_6,calendar,3,all three methods
5,target_weekday_7,calendar,3,all three methods
10,case_acceleration,local_case_history,1,one method


All three selection methods retain current cases, 7-day average cases, 7-day commuting pressure, and target-weekday controls. The 7-day and 14-day averages are highly correlated, so the absence of one feature from a selected model does not prove it has no value.

## Cell 7 — Does feature behavior change over time?

The lag analysis compares current cases with cases one or seven days earlier in May, August, and November. This is descriptive: it asks whether the relationship changes, not whether one variable causes another.

In [7]:
lag = pd.read_csv(out / 'lag_relationships_clean.csv')
lag[['month_name', 'lag', 'within_zip_slope', 'ci95_low', 'ci95_high', 'ci_includes_zero']]

,month_name,lag,within_zip_slope,ci95_low,ci95_high,ci_includes_zero
0,May,1,0.198089,-0.020698,0.288849,True
1,May,7,0.207786,0.057192,0.316171,False
2,August,1,0.087828,-0.024504,0.095107,True
3,August,7,-0.009029,-0.043360,0.030888,True
4,November,1,0.128432,-0.019841,0.196659,True
5,November,7,0.144960,0.042279,0.262517,False


### How to explain the output

August has weaker lag relationships, especially for the 7-day lag, while May and November show more positive persistence. This suggests that the usefulness of past case counts can change during different phases of the outbreak. It remains an exploratory association.

## Presentation figures

The script creates editable SVG files in `feature_analysis_outputs/figures/`. Recommended order:

1. Top features — what the models used.
2. Group ablation — what actually changed predictive error.
3. Monthly lags — how feature behavior changed over time.